# Processamento de Dados e Engenharia de Atributos

Neste jupyter, aplicaremos as transformações necessárias para preparar os dados para os algoritmos de Machine Learning. Faremos isso em etapas sequenciais para garantir o controle de qualidade dos dados.

In [2]:
# Importando bibliotecas dessa análise
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import skew
# Configuração de estilo para os gráficos 
sns.set_theme(style="whitegrid")

# Ignorar avisos 
import warnings
warnings.filterwarnings('ignore')

# Definindo o diretório 
caminho = os.getcwd()

# Carregando os datasets
df_treino = pd.read_csv(os.path.join(caminho, "arquivos", "train.csv"))
df_teste = pd.read_csv(os.path.join(caminho, "arquivos", "test.csv"))



### Etapa 3.1: Outliers, Feature Engineering e Colunas Irrelevantes
* **Remoção de Outliers:** Exclusão de imóveis com área habitável gigantesca e preços incompatíveis, que podem distorcer o modelo.
* **Engenharia de Features:** Transformação de variáveis de data em idades úteis no momento da venda (idade do imóvel e da reforma) e união de variáveis fragmentadas (agrupamento de áreas totais, soma de banheiros e varandas) para fortalecer os sinais preditivos
* **Remoção de Colunas Irrelevantes:** Descarte de variáveis com variação quase nula, excesso de valores faltantes e forte redundância matemática (multicolinearidade) ou que foram substituídas pelas novas métricas.

### Etapa 3.2: Label Encoding (Variáveis Ordinais)
* Transformação de variáveis categóricas que possuem uma hierarquia clara (ex: Excelente > Bom > Médio > Ruim) em valores numéricos. Isso permite que o modelo matemático compreenda a gradação de qualidade dos materiais e acabamentos.

In [3]:
#============================
# 3.1 (Outliers, Engenharia de Features e Colunas Irrelevantes)
#============================

# 1. Removendo Outliers Extremos (EXCLUSIVAMENTE NO TREINO)

# Filtrando as casas com mais de 4000 sqft e preço abaixo de $300.000 (Ruído extremo)

filtro_outliers = (df_treino['GrLivArea'] > 4000) & (df_treino['SalePrice'] < 300000)
df_treino = df_treino.drop(df_treino[filtro_outliers].index).reset_index(drop=True)

# 2. Função integrada de criação e remoção de colunas
def engenharia_de_features_e_limpeza(df):
    """
    Cria novas colunas (incluindo o Índice de Luxo), trata as idades do imóvel
    e remove colunas irrelevantes ou que foram substituídas no processo.
    """
    df_limpo = df.copy()
    
    # --- A. CRIAÇÃO DE NOVAS COLUNAS (FEATURE ENGINEERING) ---
    
    # Tratamento de Idades
    df_limpo['Idade_Casa'] = df_limpo['YrSold'] - df_limpo['YearBuilt']
    df_limpo['Idade_Reforma'] = df_limpo['YrSold'] - df_limpo['YearRemodAdd']
    
    # Área Total (Reduzindo a fragmentação das correlações de área)
    df_limpo['TotalArea'] = df_limpo['TotalBsmtSF'] + df_limpo['GrLivArea']
    
    # Total de Banheiros (Banheiros completos valem 1, lavabos valem 0.5)
    df_limpo['TotalBaths'] = (df_limpo['FullBath'] + (0.5 * df_limpo['HalfBath']) + 
                              df_limpo['BsmtFullBath'] + (0.5 * df_limpo['BsmtHalfBath']))
    
    # Área Total de Varanda/Porch externa
    df_limpo['TotalPorch'] = df_limpo['OpenPorchSF'] + df_limpo['EnclosedPorch'] + df_limpo['ScreenPorch']

    # Índice de Luxo (Média Ponderada)
    escala_luxo = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'NA': 0}
    df_limpo['Luxury_Score'] = (
        (df_limpo['OverallQual'] / 2 * 4) + 
        (df_limpo['KitchenQual'].map(escala_luxo).fillna(0) * 3) + 
        (df_limpo['ExterQual'].map(escala_luxo).fillna(0) * 3) + 
        (df_limpo['BsmtQual'].map(escala_luxo).fillna(0) * 2) + 
        (df_limpo['FireplaceQu'].map(escala_luxo).fillna(0) * 1)
    ) / 13

    
    # --- B. REMOÇÃO DE COLUNAS IRRELEVANTES E SUBSTITUÍDAS ---
    
    colunas_para_remover = [
        # --- 0. Colunas substituídas pelas Novas Features acima ---
        'YearBuilt', 'YearRemodAdd', 'YrSold', 'MoSold', 
        'FullBath', 'HalfBath', 'BsmtFullBath', 'BsmtHalfBath', 
        'OpenPorchSF', 'EnclosedPorch', 'ScreenPorch', 
        
        # --- 1. Variação Quase Nula (Low Variance) ---
        'Utilities', 'Street', 'Heating', 'LowQualFinSF', '3SsnPorch',
        
        # --- 2. Excesso de Valores Nulos / Esparsidade ---
        'PoolQC', 'PoolArea', 'MiscFeature', 'MiscVal', 'Alley',
        
        # --- 3. Redundância / Multicolinearidade ---
        'GarageArea',    # Redundante com 'GarageCars'
        'TotRmsAbvGrd',  # Redundante com 'GrLivArea'
        '1stFlrSF',      # Redundante com 'TotalBsmtSF'
        '2ndFlrSF',      # Embutida dentro da área total (GrLivArea)
        'GarageYrBlt',   # Redundante com as idades da casa
        
        # --- 4. Identificadores ---
        'Id'             
    ]
    
    df_limpo = df_limpo.drop(columns=colunas_para_remover, errors='ignore')
    
    return df_limpo

# 3. Aplicando a função nos dois datasets
df_treino = engenharia_de_features_e_limpeza(df_treino)
df_teste = engenharia_de_features_e_limpeza(df_teste)

print(f"Dimensões do Treino após limpeza e Feature Engineering: {df_treino.shape}")
print(f"Dimensões do Teste após limpeza e Feature Engineering: {df_teste.shape}")

Dimensões do Treino após limpeza e Feature Engineering: (1458, 60)
Dimensões do Teste após limpeza e Feature Engineering: (1459, 59)


> ### **Justificativa das Modificações e Preparação para os Modelos**
>
>
> * **Inclusão do `Luxury_Score`:** A nova feature consolida as principais notas de acabamento (`OverallQual`, `KitchenQual`, etc.) em um único índice contínuo. Como a conversão global de texto para número (Label Encoding) só ocorrerá na etapa 3.2, utilizamos um `.map()` interno apenas para viabilizar este cálculo matemático sem quebrar a estrutura do notebook.
> * **Cortes Universais:** Removemos colunas que prejudicam **qualquer** modelo, como variáveis de variância quase nula (ex: `Utilities`, onde 99% é igual), alto índice de NaNs (ex: `PoolQC`) e multicolinearidade óbvia (`GarageArea` vs `GarageCars`).
>
> **Estratégia Futura para Separação dos DataFrames:**
> * **Para o Modelo Linear (ex: Ridge/Lasso):** Utilizaremos um filtro rígido lá na frente. Selecionaremos apenas as ~10 melhores features independentes + nosso `Luxury_Score`, garantindo a ausência total de multicolinearidade. Também aplicaremos a transformação `log1p` nelas.
> * **Para o Random Forest / Gradient Boosting:** Manteremos uma base mais larga (incluindo as variáveis que sofrerão One-Hot Encoding no passo 3.4), pois modelos de árvore não sofrem com a multicolinearidade e conseguem extrair valor de interações complexas entre diversas features simultaneamente.

In [7]:
#============================
# 3.2 Label Encoding (Variáveis Ordinais)
#============================

def aplicar_label_encoding(df):
    """
    Trata valores nulos semânticos e converte variáveis categóricas ordinais 
    em numéricas usando dicionários baseados no data_description.txt.
    """
    df_encoded = df.copy()
    
    # 1. Preenchendo NaNs que possuem significado real ("Não possui") com a string 'NA'
    cols_com_na_significativo = [
        'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 
        'BsmtFinType2', 'FireplaceQu', 'GarageFinish', 'GarageQual', 
        'GarageCond', 'Fence'
    ]
    
    for col in cols_com_na_significativo:
        if col in df_encoded.columns:
            df_encoded[col] = df_encoded[col].fillna('NA')
            
    # 2. Dicionários de mapeamento
    escala_qualidade = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'NA': 0}
    
    mapeamentos = {
        # Qualidade geral e condições
        'ExterQual': escala_qualidade,
        'ExterCond': escala_qualidade,
        'BsmtQual': escala_qualidade,
        'BsmtCond': escala_qualidade,
        'HeatingQC': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1},
        'KitchenQual': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1}, 
        'FireplaceQu': escala_qualidade,
        'GarageQual': escala_qualidade,
        'GarageCond': escala_qualidade,
        
        # Outras variáveis ordinais
        'BsmtExposure': {'Gd': 4, 'Av': 3, 'Mn': 2, 'No': 1, 'NA': 0},
        'BsmtFinType1': {'GLQ': 6, 'ALQ': 5, 'BLQ': 4, 'Rec': 3, 'LwQ': 2, 'Unf': 1, 'NA': 0},
        'BsmtFinType2': {'GLQ': 6, 'ALQ': 5, 'BLQ': 4, 'Rec': 3, 'LwQ': 2, 'Unf': 1, 'NA': 0},
        'GarageFinish': {'Fin': 3, 'RFn': 2, 'Unf': 1, 'NA': 0},
        'CentralAir': {'Y': 1, 'N': 0},
        'LotShape': {'Reg': 3, 'IR1': 2, 'IR2': 1, 'IR3': 0},
        'LandSlope': {'Gtl': 2, 'Mod': 1, 'Sev': 0},
        'PavedDrive': {'Y': 2, 'P': 1, 'N': 0},
        'Fence': {'GdPrv': 4, 'MnPrv': 3, 'GdWo': 2, 'MnWw': 1, 'NA': 0}
    }
    
    # 3. Aplicando os dicionários ao dataframe
    for col, mapping in mapeamentos.items():
        if col in df_encoded.columns:
            df_encoded[col] = df_encoded[col].replace(mapping)
            # Conversão compatível com as versões recentes do Pandas
            df_encoded[col] = pd.to_numeric(df_encoded[col], errors='coerce')
            
    return df_encoded

# Executando a função nos conjuntos de Treino e Teste
df_treino = aplicar_label_encoding(df_treino)
df_teste = aplicar_label_encoding(df_teste)

print("Etapa de Label Encoding aplicada com sucesso!")

# Verificação rápida dos tipos
display(df_treino[['ExterQual', 'BsmtQual', 'CentralAir', 'LotShape']].dtypes)

Etapa de Label Encoding aplicada com sucesso!


ExterQual     int64
BsmtQual      int64
CentralAir    int64
LotShape      int64
dtype: object


> * **Preservação da Ordem Matemática:** Categorias como Qualidade da Cozinha (`KitchenQual`) variam de Excelente (`Ex`) a Ruim (`Po`). Ao mapear isso para uma escala de `5` a `1`, informamos matematicamente aos algoritmos que `5 > 4` e assim por diante. Se usássemos One-Hot Encoding (criando uma coluna para cada categoria), destruiríamos essa noção de hierarquia e explodiríamos a dimensionalidade do dataset à toa.
> * **Tratamento Inteligente de NaNs ("Nulos Semânticos"):** No Pandas, um valor `NaN` geralmente indica um erro ou dado faltante. Porém, lendo o dicionário de dados do Ames Housing, percebemos que um `NaN` na coluna `BsmtQual` (Qualidade do Porão) significa, na verdade, que *a casa não tem porão*. O código trata isso preenchendo o vazio com `NA` e, em seguida, mapeando para `0`. O modelo agora entende perfeitamente: `0` (Não tem) < `1` (Ruim) < ... < `5` (Excelente).
> 
> **Impacto na Modelagem:**
> * **Modelos Lineares:** Essa escala numérica atua como um multiplicador de peso contínuo, facilitando a interpretação dos coeficientes (ex: cada ponto extra na qualidade aumenta o preço em X dólares).
> * **Modelos Baseados em Árvore (Random Forest / XGBoost):** Permite que as árvores criem cortes (splits) lógicos contínuos, como `se KitchenQual <= 3`, dividindo eficientemente casas de qualidade média/baixa das de qualidade alta.

__________________________________________________________________________________________

### 3.3 Tratamento de Nulos Residuais

Para garantir que o modelo receba uma matriz completa, os dados com (NaN) devem ser tratados

* **Frente do Terreno (`LotFrontage`):** Imput de vazios utilizando a mediana do respectivo bairro (*Neighborhood*). Aplicamos as medianas descobertas no conjunto de treino ao conjunto de teste para evitar vazamento de dados.

* **Variáveis Categóricas e Numéricas Genéricas:** Valores textuais faltantes recebem a moda (valor mais frequente), enquanto áreas e contagens numéricas recebem zero.

### 3.4 One-Hot Encoding (Variáveis Nominais)

Transformação final das variáveis categóricas sem ordem natural (ex: Bairros, Tipo de Telhado). 
Utilizamos `pd.get_dummies` e a função de alinhamento (`align`) para assegurar que os conjuntos de treino e teste terminem exatamente com as mesmas colunas. A variável `MSSubClass`, embora numérica, representa categorias de construção e é tratada como texto.

> ### **3.3 a 3.5: Imputação, OHE, Skewness e Separação dos Datasets**
>
> Nesta etapa final de processamento, preparamos as matrizes matemáticas exatas que os algoritmos de Machine Learning irão consumir. A principal melhoria estrutural aqui é a **Bifurcação dos Dados**, criando bases otimizadas para diferentes famílias de algoritmos:
>
> * **Correção de Assimetria (Skewness):** Antes do One-Hot Encoding, aplicamos a transformação `log1p` na variável alvo (`SalePrice`) e em todas as features numéricas contínuas que apresentam forte distorção (skewness > 0.75). Isso aproxima os dados de uma distribuição normal (Gaussiana), premissa fundamental para o sucesso de Modelos Lineares (Ridge, Lasso).
> * **Dataset para Random Forest (Não-Linear):** Algoritmos baseados em árvores lidam bem com alta dimensionalidade e não sofrem com multicolinearidade. Portanto, eles receberão o DataFrame completo, contendo todas as variáveis e a expansão total do One-Hot Encoding (`get_dummies`).
> * **Dataset para Regressão Linear:** Modelos lineares sofrem com a "Maldição da Dimensionalidade" e com a redundância. Para eles, exportaremos um DataFrame enxuto, contendo apenas o `SalePrice` e as **10 features de ouro** que isolamos anteriormente (sem a poluição visual de centenas de colunas OHE).

In [8]:
import numpy as np
import pandas as pd
from scipy.stats import skew

# ==========================================
# 1. REMOÇÃO DE OUTLIERS (APENAS NO TREINO)
# ==========================================
filtro_outliers = (df_treino['GrLivArea'] > 4000) & (df_treino['SalePrice'] < 300000)
df_treino = df_treino.drop(df_treino[filtro_outliers].index).reset_index(drop=True)

# ==========================================
# 2. FUNÇÕES DE TRATAMENTO E TRANSFORMAÇÃO
# ==========================================
def tratar_nulos_finais(df_treino, df_teste):
    df_train_imp = df_treino.copy()
    df_test_imp = df_teste.copy()
    
    # Imputação de 'LotFrontage' (Mediana do Bairro)
    if 'LotFrontage' in df_train_imp.columns and 'Neighborhood' in df_train_imp.columns:
        medianas_bairro = df_train_imp.groupby('Neighborhood')['LotFrontage'].median()
        mediana_global = df_train_imp['LotFrontage'].median()
        
        df_train_imp['LotFrontage'] = df_train_imp.apply(
            lambda r: medianas_bairro.get(r['Neighborhood']) if pd.isna(r['LotFrontage']) else r['LotFrontage'], axis=1)
        df_test_imp['LotFrontage'] = df_test_imp.apply(
            lambda r: medianas_bairro.get(r['Neighborhood'], mediana_global) if pd.isna(r['LotFrontage']) else r['LotFrontage'], axis=1)

    # Imputação de Categorias com a Moda
    for col in df_train_imp.select_dtypes(include=['object']).columns:
        moda = df_train_imp[col].mode()[0]
        df_train_imp[col] = df_train_imp[col].fillna(moda)
        if col in df_test_imp.columns:
            df_test_imp[col] = df_test_imp[col].fillna(moda)

    # Imputação de Numéricos com Zero
    for col in df_train_imp.select_dtypes(exclude=['object']).columns:
        if col != 'SalePrice': 
            df_train_imp[col] = df_train_imp[col].fillna(0)
            if col in df_test_imp.columns:
                df_test_imp[col] = df_test_imp[col].fillna(0)
                
    return df_train_imp, df_test_imp

def corrigir_assimetria(df_treino, df_teste):
    """Aplica log1p no SalePrice e nas features numéricas distorcidas"""
    df_train_log = df_treino.copy()
    df_test_log = df_teste.copy()
    
    # 1. Log no Target (SalePrice)
    df_train_log['SalePrice'] = np.log1p(df_train_log['SalePrice'])
    
    # 2. Log nas features com alta assimetria (Skewness > 0.75)
    num_cols = df_train_log.select_dtypes(exclude=['object']).columns.drop('SalePrice')
    assimetria = df_train_log[num_cols].apply(lambda x: skew(x.dropna())).sort_values(ascending=False)
    features_tortas = assimetria[abs(assimetria) > 0.75].index
    
    for col in features_tortas:
        df_train_log[col] = np.log1p(df_train_log[col])
        if col in df_test_log.columns:
            df_test_log[col] = np.log1p(df_test_log[col])
            
    return df_train_log, df_test_log

def aplicar_ohe_e_bifurcar(df_treino, df_teste):
    df_train_ohe = df_treino.copy()
    df_test_ohe = df_teste.copy()
    
    # Correção: MSSubClass é categórica
    if 'MSSubClass' in df_train_ohe.columns:
        df_train_ohe['MSSubClass'] = df_train_ohe['MSSubClass'].astype(str)
        df_test_ohe['MSSubClass'] = df_test_ohe['MSSubClass'].astype(str)
        
    alvo = df_train_ohe.pop('SalePrice')
    
    # Aplicação do OHE
    df_train_ohe = pd.get_dummies(df_train_ohe, drop_first=True, dtype=int)
    df_test_ohe = pd.get_dummies(df_test_ohe, drop_first=True, dtype=int)
    df_train_ohe, df_test_ohe = df_train_ohe.align(df_test_ohe, join='left', axis=1, fill_value=0)
    
    df_train_ohe['SalePrice'] = alvo
    
    # ==========================================
    # SEPARAÇÃO: LINEAR vs RANDOM FOREST
    # ==========================================
    
    # 1. Dataset Completo para Random Forest (Com OHE)
    df_treino_tree = df_train_ohe.copy()
    df_teste_tree = df_test_ohe.copy()
    
    # 2. Dataset Enxuto para Modelo Linear (As 10 isoladas + SalePrice)
    features_lineares = [
        'Luxury_Score', 'GrLivArea', 'GarageCars', 'TotalBsmtSF', 
        'TotalBaths', 'Idade_Casa', 'Idade_Reforma', 'MasVnrArea', 
        'Fireplaces', 'LotFrontage'
    ]
    
    # Filtra mantendo apenas o que existe + Target
    cols_lin_treino = [c for c in features_lineares if c in df_train_ohe.columns] + ['SalePrice']
    cols_lin_teste = [c for c in features_lineares if c in df_test_ohe.columns]
    
    df_treino_linear = df_train_ohe[cols_lin_treino].copy()
    df_teste_linear = df_test_ohe[cols_lin_teste].copy()
    
    return df_treino_linear, df_teste_linear, df_treino_tree, df_teste_tree

# ==========================================
# 3. EXECUÇÃO DO PIPELINE
# ==========================================
df_treino, df_teste = tratar_nulos_finais(df_treino, df_teste)
df_treino, df_teste = corrigir_assimetria(df_treino, df_teste)

df_treino_linear, df_teste_linear, df_treino_tree, df_teste_tree = aplicar_ohe_e_bifurcar(df_treino, df_teste)

print("--- MODELOS BASEADOS EM ÁRVORES (OHE Completo) ---")
print(f"Treino Árvore: {df_treino_tree.shape} | Teste Árvore: {df_teste_tree.shape}")
print(f"Nulos residuais: {df_treino_tree.isnull().sum().sum()}\n")

print("--- MODELOS LINEARES (10 Features de Ouro) ---")
print(f"Treino Linear: {df_treino_linear.shape} | Teste Linear: {df_teste_linear.shape}")

--- MODELOS BASEADOS EM ÁRVORES (OHE Completo) ---
Treino Árvore: (1458, 190) | Teste Árvore: (1459, 189)
Nulos residuais: 0

--- MODELOS LINEARES (10 Features de Ouro) ---
Treino Linear: (1458, 11) | Teste Linear: (1459, 10)


In [15]:
print(df_treino_linear.columns)
print("\n\n\n\n")
print(df_treino_tree.columns)

Index(['Luxury_Score', 'GrLivArea', 'GarageCars', 'TotalBsmtSF', 'TotalBaths',
       'Idade_Casa', 'Idade_Reforma', 'MasVnrArea', 'Fireplaces',
       'LotFrontage', 'SalePrice'],
      dtype='str')





Index(['LotFrontage', 'LotArea', 'LotShape', 'LandSlope', 'OverallQual',
       'OverallCond', 'MasVnrArea', 'ExterQual', 'ExterCond', 'BsmtQual',
       ...
       'SaleType_ConLw', 'SaleType_New', 'SaleType_Oth', 'SaleType_WD',
       'SaleCondition_AdjLand', 'SaleCondition_Alloca', 'SaleCondition_Family',
       'SaleCondition_Normal', 'SaleCondition_Partial', 'SalePrice'],
      dtype='str', length=190)


### (3.6)  Exportando os **Dataframes finais**, que serão usados nos modelos, para *`csv`*

In [16]:
# ==========================================
# 4. EXPORTANDO DATAFRAMES FINAIS PARA CSV
# ==========================================
import os
import pandas as pd

caminho = os.getcwd()

# 1. Recuperando os IDs originais do Teste (Para garantir os 1459 exatos)
df_teste_bruto = pd.read_csv(os.path.join(caminho, "arquivos", "test.csv"))
test_ids = df_teste_bruto[['Id']] # Guardamos como DataFrame

# 2. Verificação de Segurança 
print("--- Verificação de Linhas ---")
print(f"Treino Linear (1458 esperadas): {df_treino_linear.shape[0]}")
print(f"Teste Linear  (1459 esperadas): {df_teste_linear.shape[0]}")
print(f"Treino Tree   (1458 esperadas): {df_treino_tree.shape[0]}")
print(f"Teste Tree    (1459 esperadas): {df_teste_tree.shape[0]}\n")

# 3. Garantindo que o diretório de saída existe
pasta_destino = os.path.join(caminho, "csv_gerados")
os.makedirs(pasta_destino, exist_ok=True)

# 4. Exportando os DataFrames tratados - MODELOS LINEARES
df_treino_linear.to_csv(os.path.join(pasta_destino, "df_treino_linear.csv"), index=False)
df_teste_linear.to_csv(os.path.join(pasta_destino, "df_teste_linear.csv"), index=False)

# 5. Exportando os DataFrames tratados - MODELOS DE ÁRVORE (RANDOM FOREST / XGBOOST)
df_treino_tree.to_csv(os.path.join(pasta_destino, "df_treino_tree.csv"), index=False)
df_teste_tree.to_csv(os.path.join(pasta_destino, "df_teste_tree.csv"), index=False)

# 6. Exportando o arquivo exclusivo com os IDs
test_ids.to_csv(os.path.join(pasta_destino, "test_ids.csv"), index=False)

print("Todos os 5 arquivos CSV foram exportados com sucesso na pasta 'csv_gerados'!")

--- Verificação de Linhas ---
Treino Linear (1458 esperadas): 1458
Teste Linear  (1459 esperadas): 1459
Treino Tree   (1458 esperadas): 1458
Teste Tree    (1459 esperadas): 1459

Todos os 5 arquivos CSV foram exportados com sucesso na pasta 'csv_gerados'!
